# W03 · Grid bottleneck and paper Figure 5
# W03 · Grid 瓶頸與論文 Figure 5

The paper sweeps **all 4×4 combinations** of grid resolutions
`[16,32,64,128]` and feature widths `[8,16,32,64]` for BI-grid, LPE,
and three-frequency Grid-PEPS on native-4K images with L1. The paper does
not identify those images, optimizer, batch size, or training steps, so an
exact claim is blocked until a checksum manifest and explicit assumptions
are supplied. `course_fast` remains a separate, runnable smoke path.

論文對 BI-grid、LPE、三頻 Grid-PEPS 執行 4×4 解析度/特徵寬度 sweep。
由於論文未公開影像清單與完整訓練預算，本 notebook 不會拿 Kodak 或論文數字
冒充本機 Figure 5 結果。

In [ ]:
# Repo bootstrap: make `peps` and `apps` importable from the notebook.
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
from peps.train import auto_device
device = auto_device()
print('torch', torch.__version__, '| device', device)

## 1. Choose a profile / 選擇執行軌

In [ ]:
import subprocess, json
PROFILE = os.environ.get('PEPS_PROFILE', 'course_fast')
if PROFILE not in {'course_fast', 'paper_exact'}:
    raise ValueError('PEPS_PROFILE must be course_fast or paper_exact')
print('profile:', PROFILE)

## 2. Structural sweep oracle / Sweep 結構 oracle
This builds the exact matrix and checks dimensions only; it does not emit
quality numbers.

In [ ]:
from apps.image.build import build_paper_fig5
matrix = []
for method in ('bi_grid', 'lpe', 'grid_peps'):
    for resolution in (16, 32, 64, 128):
        for feature_dim in (8, 16, 32, 64):
            model, params = build_paper_fig5(method, resolution=resolution, feature_dim=feature_dim)
            matrix.append((method, resolution, feature_dim, params))
print('matrix entries:', len(matrix))
assert len(matrix) == 3 * 4 * 4

## 3. Machine-readable readiness / 機器可讀 readiness

In [ ]:
check_cmd = [sys.executable, '-m', 'experiments.reproduce', 'check',
             '--profile', PROFILE, '--artifact', 'image-fig5']
if os.environ.get('FIG5_MANIFEST'):
    check_cmd += ['--fig5-manifest', os.environ['FIG5_MANIFEST']]
checked = subprocess.run(check_cmd, text=True, capture_output=True)
print(checked.stdout)

## 4. Run with provenance / 以 provenance 執行
`course_fast` executes a real two-step image optimization. `paper_exact`
requires `FIG5_MANIFEST`, `FIG5_STEPS`, and explicit opt-in; its manifest
will remain labelled `protocol_assumption`.

In [ ]:
if PROFILE == 'course_fast':
    run_cmd = [sys.executable, '-m', 'experiments.reproduce', 'smoke', '--task', 'image']
elif os.environ.get('RUN_PAPER_EXACT') == '1':
    run_cmd = [sys.executable, '-m', 'experiments.reproduce', 'run',
               '--artifact', 'image-fig5', '--fig5-manifest', os.environ['FIG5_MANIFEST'],
               '--assumed-steps', os.environ['FIG5_STEPS'], '--allow-protocol-assumptions']
else:
    run_cmd = None
    print('Paper run not started; exact dataset/training details are unavailable.')
if run_cmd:
    completed = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
    print(completed.stdout)

## 5. Result contract / 結果契約
Use only `summary.csv` beside the printed `manifest.json`; never import a
legacy CSV or copy the curve from the publication.